# 🚬 흡연 분류 V15 - V12 Fixed + V4 Core + AutoML

## 핵심 원칙
- ✅ **OOF(CV)로만 채택/폐기** - 추측 금지
- ✅ **V12 결함 제거**: 인코딩은 fold train fit → valid/test transform
- ✅ **V4 코어 모듈**: 옵션 ON/OFF + ablation
- ✅ **FLAML AutoML**: outer 5-fold OOF 생성

---

## CONFIG (상단 토글)

In [ ]:
#===========================================
# 상단 CONFIG
#===========================================
SEED = 42
N_SPLITS = 5

# 전처리 옵션
MISSING_METHOD = "B"  # A: fillna(0), B: median/mode, C: B+_isna
USE_OUTLIER_CLIP = True
OUTLIER_Q_LOW = 0.01
OUTLIER_Q_HIGH = 0.99

# 피처 그룹 ON/OFF
USE_G1_RATIO = True
USE_G2_LOG_SQ = True
USE_G3_HIGH_FLAG = True
USE_G4_INTERACTION = True
USE_G5_BINNING = True
HIGH_QUANTILE = 0.80

# 스케일링 (XGB/LGB/AutoML용)
USE_SCALER = False

# 모델 옵션
USE_RF = True  # RandomForest 포함 여부
TUNE_BUDGET = None  # 튜닝 시간(초), None이면 OFF

# AutoML 옵션
USE_AUTOML = True
AUTOML_ENGINE = "flaml"  # flaml(기본) or autogluon
AUTOML_TIME_BUDGET = 120  # 초/fold

print("✅ CONFIG 설정 완료")

## STEP 0: 환경 설정

In [ ]:
!pip install -q xgboost lightgbm catboost flaml

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from flaml import AutoML

import random
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
set_seed(SEED)

# 경로 고정
base_path = '/content/drive/MyDrive/AI_Projects/smoking_hackathon/'
train_path = base_path + 'data/train.csv'
test_path = base_path + 'data/test.csv'
submission_path = base_path + 'data/sample_submission.csv'
result_path = base_path + 'results/'

os.makedirs(result_path, exist_ok=True)
print(f"✅ result_path: {result_path}")

In [ ]:
#===========================================
# 안전장치 1: 파일 존재 여부
#===========================================
print("=" * 60)
print("🔍 안전장치 1: 파일 존재 여부")
print("=" * 60)

files_check = {
    'train.csv': train_path,
    'test.csv': test_path,
    'sample_submission.csv': submission_path
}
all_exist = True
for name, path in files_check.items():
    exists = os.path.exists(path)
    print(f"   {name}: {'OK' if exists else 'NOT FOUND'}")
    if not exists:
        all_exist = False

if not all_exist:
    raise FileNotFoundError("필수 파일이 없습니다.")
print("\n✅ 모든 파일 존재 확인")

## STEP 1: 데이터 로드 + 검증

In [ ]:
train_raw = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)
submission = pd.read_csv(submission_path)

print("=" * 60)
print("📊 데이터 로드")
print("=" * 60)
print(f"Train: {train_raw.shape}, Test: {test_raw.shape}")

In [ ]:
#===========================================
# 안전장치 2: 컬럼/중복/결측치
#===========================================
print("\n🔍 안전장치 2: 컬럼/중복/결측치")
print(f"\nTrain 컬럼: {train_raw.columns.tolist()}")
print(f"Test 컬럼: {test_raw.columns.tolist()}")

train_dup = train_raw.columns[train_raw.columns.duplicated()].tolist()
test_dup = test_raw.columns[test_raw.columns.duplicated()].tolist()
print(f"\n중복 컬럼 - Train: {train_dup if train_dup else '없음'}, Test: {test_dup if test_dup else '없음'}")

print("\n결측치 비율 (상위 20):")
missing = (train_raw.isnull().sum() / len(train_raw) * 100).sort_values(ascending=False)
print(missing.head(20).to_string())

## STEP 2: 컬럼 매핑

In [ ]:
COL_MAP = {
    'ID': 'id', '나이': 'age', '키(cm)': 'height_cm', '몸무게(kg)': 'weight_kg',
    'BMI': 'bmi', '시력(좌)': 'eyesight_left', '시력(우)': 'eyesight_right',
    '청력(좌)': 'hearing_left', '청력(우)': 'hearing_right', '충치': 'cavity',
    '공복 혈당': 'fasting_glucose', '수축기 혈압': 'systolic_bp', '이완기 혈압': 'diastolic_bp',
    '중성 지방': 'triglyceride', '혈청 크레아티닌': 'serum_creatinine',
    '콜레스테롤': 'cholesterol', '고밀도 지단백': 'hdl', '고밀도지단백': 'hdl',
    '저밀도 지단백': 'ldl', '저밀도지단백': 'ldl', '헤모글로빈': 'hemoglobin',
    '요 단백': 'urine_protein', '간 효소율': 'gtp', 'label': 'label'
}

def rename_cols(df):
    return df.rename(columns={k: v for k, v in COL_MAP.items() if k in df.columns})

train = rename_cols(train_raw.copy())
test = rename_cols(test_raw.copy())

# ID 분리
test_ids = test['id'].copy() if 'id' in test.columns else pd.Series(range(len(test)))
train = train.drop('id', axis=1, errors='ignore')
test = test.drop('id', axis=1, errors='ignore')

# X, y 분리
y = train['label'].copy()
X = train.drop('label', axis=1)
X_test_orig = test.drop('label', axis=1, errors='ignore')

print(f"\n✅ X: {X.shape}, y: {y.shape}, X_test: {X_test_orig.shape}")
print(f"클래스 분포: {y.value_counts().to_dict()}")

## STEP 3: 전처리/피처 함수 정의

In [ ]:
# 범주형 컬럼 정의
CAT_COLS_BASE = ['cavity', 'urine_protein']
CAT_COLS_DERIVED = ['age_group', 'bmi_group']

def handle_missing(df_tr, df_va, df_te, method):
    """결측치 처리 - train 기준 fit"""
    tr, va, te = df_tr.copy(), df_va.copy(), df_te.copy()
    isna_cols = []
    
    if method == "A":
        tr = tr.fillna(0)
        va = va.fillna(0)
        te = te.fillna(0)
    elif method in ["B", "C"]:
        fill_vals = {}
        for col in tr.columns:
            if tr[col].dtype in ['float64', 'int64', 'float32', 'int32']:
                fill_vals[col] = tr[col].median()
            else:
                mode_vals = tr[col].mode()
                fill_vals[col] = mode_vals.iloc[0] if len(mode_vals) > 0 else 0
        
        for col, val in fill_vals.items():
            tr[col] = tr[col].fillna(val)
            va[col] = va[col].fillna(val)
            te[col] = te[col].fillna(val)
        
        if method == "C":
            for col in df_tr.columns:
                if df_tr[col].isnull().sum() > 0:
                    flag = f"{col}_isna"
                    tr[flag] = df_tr[col].isnull().astype(int)
                    va[flag] = df_va[col].isnull().astype(int)
                    te[flag] = df_te[col].isnull().astype(int)
                    isna_cols.append(flag)
    
    return tr, va, te, isna_cols


def clip_outliers(df_tr, df_va, df_te, q_low, q_high):
    """이상치 클리핑 - train 기준 분위수"""
    tr, va, te = df_tr.copy(), df_va.copy(), df_te.copy()
    clip_info = {}
    
    for col in tr.select_dtypes(include=[np.number]).columns:
        lower = tr[col].quantile(q_low)
        upper = tr[col].quantile(q_high)
        clip_info[col] = (lower, upper)
        tr[col] = tr[col].clip(lower, upper)
        va[col] = va[col].clip(lower, upper)
        te[col] = te[col].clip(lower, upper)
    
    return tr, va, te, clip_info


def create_features(df, ref_df=None, high_q=0.80):
    """피처 엔지니어링 - ref_df 기준 분위수 계산"""
    df = df.copy()
    ref = ref_df if ref_df is not None else df
    cols = df.columns.tolist()
    created = []
    thresholds_used = {}
    
    # G1: 비율
    if USE_G1_RATIO:
        if 'triglyceride' in cols and 'hdl' in cols:
            df['tg_hdl_ratio'] = df['triglyceride'] / (df['hdl'] + 1)
            created.append('tg_hdl_ratio')
        if 'hdl' in cols and 'ldl' in cols:
            df['hdl_ldl_ratio'] = df['hdl'] / (df['ldl'] + 1)
            created.append('hdl_ldl_ratio')
    
    # G2: 로그/제곱
    if USE_G2_LOG_SQ:
        if 'gtp' in cols:
            df['gtp_log'] = np.log1p(df['gtp'])
            created.append('gtp_log')
        if 'triglyceride' in cols:
            df['tg_log'] = np.log1p(df['triglyceride'])
            created.append('tg_log')
        if 'hemoglobin' in cols:
            df['hemo_sq'] = df['hemoglobin'] ** 2
            created.append('hemo_sq')
    
    # G3: high 플래그 (분위수 기반)
    if USE_G3_HIGH_FLAG:
        for col in ['hemoglobin', 'gtp', 'triglyceride']:
            if col in cols:
                th = ref[col].quantile(high_q)
                thresholds_used[f"{col}_high"] = th
                df[f"{col}_high"] = (df[col] > th).astype(int)
                created.append(f"{col}_high")
    
    # G4: 상호작용
    if USE_G4_INTERACTION:
        if 'hemoglobin' in cols and 'gtp' in cols:
            df['hemo_x_gtp'] = df['hemoglobin'] * df['gtp']
            created.append('hemo_x_gtp')
        if 'age' in cols and 'hemoglobin' in cols:
            df['age_x_hemo'] = df['age'] * df['hemoglobin']
            created.append('age_x_hemo')
    
    # G5: 구간화 (분위수 기반)
    if USE_G5_BINNING:
        if 'age' in cols:
            bins = [0] + list(ref['age'].quantile([0.25, 0.5, 0.75]).values) + [200]
            df['age_group'] = pd.cut(df['age'], bins=bins, labels=['0','1','2','3'], duplicates='drop')
            df['age_group'] = df['age_group'].astype(str).replace('nan', '0')
            created.append('age_group')
        if 'bmi' in cols:
            bins = [0, 18.5, 23, 25, 30, 100]
            df['bmi_group'] = pd.cut(df['bmi'], bins=bins, labels=['0','1','2','3','4'], duplicates='drop')
            df['bmi_group'] = df['bmi_group'].astype(str).replace('nan', '0')
            created.append('bmi_group')
    
    df = df.fillna(0).replace([np.inf, -np.inf], 0)
    return df, created, thresholds_used

print("✅ 전처리/피처 함수 정의 완료")

In [ ]:
#===========================================
# 안전장치 3: 파생피처 상수 여부 체크
#===========================================
print("\n🔍 안전장치 3: 파생피처 상수 여부 체크")

X_temp, created_temp, th_temp = create_features(X.copy(), ref_df=X, high_q=HIGH_QUANTILE)
print(f"\n생성된 피처: {created_temp}")
print(f"사용된 임계값: {th_temp}")

const_features = []
for col in created_temp:
    if col in X_temp.columns:
        nunique = X_temp[col].nunique()
        if nunique <= 1:
            const_features.append(col)
            print(f"   ⚠️ {col}: 상수 피처 (nunique={nunique})")
        else:
            print(f"   ✅ {col}: {nunique} unique")

if const_features:
    print(f"\n⚠️ 상수 피처 자동 제거 대상: {const_features}")

## STEP 4: CV 설정 + OOF 생성

In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# OOF 저장
oof_xgb = np.zeros(len(X))
oof_lgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))
oof_rf = np.zeros(len(X)) if USE_RF else None
oof_automl = np.zeros(len(X)) if USE_AUTOML else None

# Test 예측
test_xgb = np.zeros(len(X_test_orig))
test_lgb = np.zeros(len(X_test_orig))
test_cat = np.zeros(len(X_test_orig))
test_rf = np.zeros(len(X_test_orig)) if USE_RF else None
test_automl = np.zeros(len(X_test_orig)) if USE_AUTOML else None

fold_results = []

print(f"\n✅ CV: StratifiedKFold(n_splits={N_SPLITS}, shuffle=True, random_state={SEED})")

In [ ]:
print("\n" + "=" * 60)
print("🔄 STEP 4: OOF 생성 (Baseline + AutoML)")
print("=" * 60)

for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y)):
    print(f"\n--- Fold {fold+1}/{N_SPLITS} ---")
    
    X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
    X_te = X_test_orig.copy()
    
    # 1) 결측치 처리
    X_tr, X_va, X_te, isna_cols = handle_missing(X_tr, X_va, X_te, MISSING_METHOD)
    
    # 2) 이상치 클리핑
    if USE_OUTLIER_CLIP:
        X_tr, X_va, X_te, clip_info = clip_outliers(X_tr, X_va, X_te, OUTLIER_Q_LOW, OUTLIER_Q_HIGH)
    
    # 3) 피처 엔지니어링 (train 기준)
    X_tr, created, thresholds = create_features(X_tr, ref_df=X_tr, high_q=HIGH_QUANTILE)
    X_va, _, _ = create_features(X_va, ref_df=X_tr, high_q=HIGH_QUANTILE)
    X_te, _, _ = create_features(X_te, ref_df=X_tr, high_q=HIGH_QUANTILE)
    
    if fold == 0:
        print(f"   피처 수: {X_tr.shape[1]}, 임계값: {thresholds}")
    
    # 4) 상수 피처 제거
    for col in const_features:
        X_tr = X_tr.drop(col, axis=1, errors='ignore')
        X_va = X_va.drop(col, axis=1, errors='ignore')
        X_te = X_te.drop(col, axis=1, errors='ignore')
    
    # 컬럼 정렬
    common_cols = sorted([c for c in X_tr.columns if c in X_te.columns])
    X_tr = X_tr[common_cols]
    X_va = X_va[common_cols]
    X_te = X_te[common_cols]
    
    #===========================================
    # 안전장치 4: 범주형 인코딩 (train fit → valid/test transform)
    #===========================================
    cat_cols_actual = [c for c in CAT_COLS_BASE + CAT_COLS_DERIVED if c in X_tr.columns]
    num_cols = [c for c in X_tr.columns if c not in cat_cols_actual]
    
    # CatBoost용: str 변환
    X_tr_cat = X_tr.copy()
    X_va_cat = X_va.copy()
    X_te_cat = X_te.copy()
    for col in cat_cols_actual:
        X_tr_cat[col] = X_tr_cat[col].astype(str)
        X_va_cat[col] = X_va_cat[col].astype(str)
        X_te_cat[col] = X_te_cat[col].astype(str)
    
    # XGB/LGB/RF/AutoML용: OrdinalEncoder (train fit → transform)
    X_tr_enc = X_tr.copy()
    X_va_enc = X_va.copy()
    X_te_enc = X_te.copy()
    
    if cat_cols_actual:
        enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        X_tr_enc[cat_cols_actual] = enc.fit_transform(X_tr[cat_cols_actual].astype(str))
        X_va_enc[cat_cols_actual] = enc.transform(X_va[cat_cols_actual].astype(str))
        X_te_enc[cat_cols_actual] = enc.transform(X_te[cat_cols_actual].astype(str))
    
    # 스케일링 (옵션)
    if USE_SCALER:
        scaler = StandardScaler()
        X_tr_enc[num_cols] = scaler.fit_transform(X_tr_enc[num_cols])
        X_va_enc[num_cols] = scaler.transform(X_va_enc[num_cols])
        X_te_enc[num_cols] = scaler.transform(X_te_enc[num_cols])
    
    #===========================================
    # 모델 학습
    #===========================================
    # XGBoost
    xgb_model = XGBClassifier(
        n_estimators=500, max_depth=5, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, verbosity=0, use_label_encoder=False, eval_metric='logloss'
    )
    xgb_model.fit(X_tr_enc, y_tr)
    oof_xgb[va_idx] = xgb_model.predict_proba(X_va_enc)[:, 1]
    test_xgb += xgb_model.predict_proba(X_te_enc)[:, 1] / N_SPLITS
    
    # LightGBM
    lgb_model = LGBMClassifier(
        n_estimators=500, max_depth=5, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8,
        class_weight='balanced', random_state=SEED, verbose=-1
    )
    lgb_model.fit(X_tr_enc, y_tr)
    oof_lgb[va_idx] = lgb_model.predict_proba(X_va_enc)[:, 1]
    test_lgb += lgb_model.predict_proba(X_te_enc)[:, 1] / N_SPLITS
    
    # CatBoost
    cat_model = CatBoostClassifier(
        iterations=500, depth=5, learning_rate=0.03,
        auto_class_weights='Balanced', random_state=SEED, verbose=0,
        cat_features=cat_cols_actual if cat_cols_actual else None
    )
    cat_model.fit(X_tr_cat, y_tr)
    oof_cat[va_idx] = cat_model.predict_proba(X_va_cat)[:, 1]
    test_cat += cat_model.predict_proba(X_te_cat)[:, 1] / N_SPLITS
    
    # RandomForest (옵션)
    if USE_RF:
        rf_model = RandomForestClassifier(
            n_estimators=300, max_depth=15, min_samples_split=5,
            class_weight='balanced', random_state=SEED, n_jobs=-1
        )
        rf_model.fit(X_tr_enc, y_tr)
        oof_rf[va_idx] = rf_model.predict_proba(X_va_enc)[:, 1]
        test_rf += rf_model.predict_proba(X_te_enc)[:, 1] / N_SPLITS
    
    # AutoML (FLAML)
    if USE_AUTOML:
        automl = AutoML()
        automl.fit(
            X_tr_enc, y_tr,
            task='classification', metric='accuracy',
            time_budget=AUTOML_TIME_BUDGET, verbose=0
        )
        oof_automl[va_idx] = automl.predict_proba(X_va_enc)[:, 1]
        test_automl += automl.predict_proba(X_te_enc)[:, 1] / N_SPLITS
        print(f"   AutoML best: {automl.best_estimator}")
    
    # Fold 결과 (참고용)
    xgb_acc = accuracy_score(y_va, (oof_xgb[va_idx] >= 0.5).astype(int))
    lgb_acc = accuracy_score(y_va, (oof_lgb[va_idx] >= 0.5).astype(int))
    cat_acc = accuracy_score(y_va, (oof_cat[va_idx] >= 0.5).astype(int))
    rf_acc = accuracy_score(y_va, (oof_rf[va_idx] >= 0.5).astype(int)) if USE_RF else 0
    automl_acc = accuracy_score(y_va, (oof_automl[va_idx] >= 0.5).astype(int)) if USE_AUTOML else 0
    
    print(f"   XGB:{xgb_acc:.4f} LGB:{lgb_acc:.4f} CAT:{cat_acc:.4f}", end="")
    if USE_RF:
        print(f" RF:{rf_acc:.4f}", end="")
    if USE_AUTOML:
        print(f" AutoML:{automl_acc:.4f}", end="")
    print()

print("\n✅ OOF 생성 완료!")

## STEP 5: OOF Threshold Sweep + 비교

In [ ]:
def threshold_sweep(oof_proba, y_true, name):
    """임계값 sweep → best_t, best_acc 반환"""
    best_t, best_acc = 0.5, 0
    for t in np.arange(0.30, 0.71, 0.01):
        acc = accuracy_score(y_true, (oof_proba >= t).astype(int))
        if acc > best_acc:
            best_acc = acc
            best_t = round(t, 2)
    return name, best_t, best_acc

print("\n" + "=" * 60)
print("📊 STEP 5: OOF Threshold Sweep")
print("=" * 60)

results = []

# 개별 모델
results.append(threshold_sweep(oof_xgb, y, 'XGBoost'))
results.append(threshold_sweep(oof_lgb, y, 'LightGBM'))
results.append(threshold_sweep(oof_cat, y, 'CatBoost'))
if USE_RF:
    results.append(threshold_sweep(oof_rf, y, 'RandomForest'))

# 앙상블: 단순 평균
if USE_RF:
    oof_avg = (oof_xgb + oof_lgb + oof_cat + oof_rf) / 4
    test_avg = (test_xgb + test_lgb + test_cat + test_rf) / 4
else:
    oof_avg = (oof_xgb + oof_lgb + oof_cat) / 3
    test_avg = (test_xgb + test_lgb + test_cat) / 3
results.append(threshold_sweep(oof_avg, y, 'Ensemble_Avg'))

# 앙상블: 가중 평균 후보
weight_candidates = [
    ('W1', 0.35, 0.35, 0.30, 0.0),
    ('W2', 0.40, 0.30, 0.30, 0.0),
    ('W3', 0.30, 0.35, 0.25, 0.10),
]

best_ens_name, best_ens_oof, best_ens_test = 'Ensemble_Avg', oof_avg, test_avg
best_ens_result = results[-1]

for wname, w1, w2, w3, w4 in weight_candidates:
    if USE_RF and w4 > 0:
        oof_w = w1*oof_xgb + w2*oof_lgb + w3*oof_cat + w4*oof_rf
        test_w = w1*test_xgb + w2*test_lgb + w3*test_cat + w4*test_rf
    else:
        w_sum = w1 + w2 + w3
        oof_w = (w1*oof_xgb + w2*oof_lgb + w3*oof_cat) / w_sum
        test_w = (w1*test_xgb + w2*test_lgb + w3*test_cat) / w_sum
    
    name, t, acc = threshold_sweep(oof_w, y, f'Ensemble_{wname}')
    results.append((name, t, acc))
    
    if acc > best_ens_result[2]:
        best_ens_result = (name, t, acc)
        best_ens_oof = oof_w
        best_ens_test = test_w
        best_ens_name = name

# AutoML
if USE_AUTOML:
    results.append(threshold_sweep(oof_automl, y, 'AutoML_FLAML'))

# 결과 표
results_df = pd.DataFrame(results, columns=['Method', 'Best_Threshold', 'Best_OOF_Acc'])
results_df = results_df.sort_values('Best_OOF_Acc', ascending=False)
print("\n" + results_df.to_string(index=False))

In [ ]:
# 최종 채택
print("\n" + "=" * 60)
print("🏆 최종 채택")
print("=" * 60)

top_result = results_df.iloc[0]
CHOSEN_METHOD = top_result['Method']
BEST_T = top_result['Best_Threshold']
BEST_ACC = top_result['Best_OOF_Acc']

# 선택된 OOF/Test
if 'AutoML' in CHOSEN_METHOD:
    CHOSEN_OOF = oof_automl
    CHOSEN_TEST = test_automl
elif 'Ensemble' in CHOSEN_METHOD:
    CHOSEN_OOF = best_ens_oof
    CHOSEN_TEST = best_ens_test
elif CHOSEN_METHOD == 'XGBoost':
    CHOSEN_OOF = oof_xgb
    CHOSEN_TEST = test_xgb
elif CHOSEN_METHOD == 'LightGBM':
    CHOSEN_OOF = oof_lgb
    CHOSEN_TEST = test_lgb
elif CHOSEN_METHOD == 'CatBoost':
    CHOSEN_OOF = oof_cat
    CHOSEN_TEST = test_cat
elif CHOSEN_METHOD == 'RandomForest':
    CHOSEN_OOF = oof_rf
    CHOSEN_TEST = test_rf
else:
    CHOSEN_OOF = best_ens_oof
    CHOSEN_TEST = best_ens_test

print(f"\n채택: {CHOSEN_METHOD}")
print(f"Best Threshold: {BEST_T}")
print(f"Best OOF Accuracy: {BEST_ACC:.5f}")

# OOF 비교표 저장
results_df.to_csv(result_path + 'oof_summary_v15.csv', index=False)
print(f"\n✅ oof_summary_v15.csv 저장 완료")

## STEP 6: 제출 파일 생성

In [ ]:
print("\n" + "=" * 60)
print("📁 STEP 6: 제출 파일 생성")
print("=" * 60)

thresholds_to_save = [
    round(BEST_T - 0.02, 2),
    round(BEST_T, 2),
    round(BEST_T + 0.02, 2)
]

saved_files = []
method_short = CHOSEN_METHOD.lower().replace('_', '').replace(' ', '')[:10]

for t in thresholds_to_save:
    pred = (CHOSEN_TEST >= t).astype(int)
    oof_pred = (CHOSEN_OOF >= t).astype(int)
    oof_acc = accuracy_score(y, oof_pred)
    
    sub = submission.copy()
    sub['label'] = pred
    sub['label'] = sub['label'].astype(int)
    
    t_str = str(int(t * 100)).zfill(2)
    if t == BEST_T:
        fname = f'submission_v15_{method_short}_t{t_str}_best.csv'
    else:
        fname = f'submission_v15_{method_short}_t{t_str}.csv'
    
    fpath = result_path + fname
    sub.to_csv(fpath, index=False)
    saved_files.append(fname)
    
    n1 = (pred == 1).sum()
    marker = "⭐" if t == BEST_T else "  "
    print(f"{marker} {fname}: t={t:.2f}, OOF_Acc={oof_acc:.5f}, 흡연={n1} ({n1/len(pred)*100:.1f}%)")

print(f"\n✅ {len(saved_files)}개 파일 저장 완료")

In [ ]:
# 저장된 파일 목록
print("\n📁 저장된 파일:")
for f in saved_files:
    print(f"   ✅ {result_path}{f}")
print(f"   ✅ {result_path}oof_summary_v15.csv")

## STEP 7: 최종 요약

In [ ]:
print("\n" + "=" * 60)
print("🎉 V15 완료")
print("=" * 60)

print(f"\n채택: {CHOSEN_METHOD}")
print(f"Best Threshold: {BEST_T}")
print(f"Best OOF Accuracy: {BEST_ACC:.5f}")

print(f"\n설정:")
print(f"   Missing: {MISSING_METHOD}")
print(f"   Outlier Clip: {USE_OUTLIER_CLIP} ({OUTLIER_Q_LOW}-{OUTLIER_Q_HIGH})")
print(f"   Features: G1={USE_G1_RATIO}, G2={USE_G2_LOG_SQ}, G3={USE_G3_HIGH_FLAG}, G4={USE_G4_INTERACTION}, G5={USE_G5_BINNING}")
print(f"   RF: {USE_RF}, AutoML: {USE_AUTOML}")

print(f"\n🏆 최종: method={CHOSEN_METHOD} / OOF={BEST_ACC:.5f} / best_t={BEST_T}")

In [ ]:
from google.colab import files

best_fname = f'submission_v15_{method_short}_t{str(int(BEST_T*100)).zfill(2)}_best.csv'
files.download(result_path + best_fname)
print(f"\n📥 다운로드: {best_fname}")

In [ ]:
print("\n📥 추가 다운로드:")
for f in saved_files:
    if 'best' not in f:
        files.download(result_path + f)
        print(f"   ✅ {f}")